# 10年定着予測 - TF-IDFの次元調整・頑健性確認 と 有望ブロックの組み合わせ

**背景**: `16_feature_ablation`で、テキストTF-IDF単体（A_only）がbaselineを検証Log Lossで4.15%改善し、
実際にPublicスコアも更新した（0.563076→0.559570）。しかし検証での改善幅(4.15%)に対しPublicでの実際の
改善(0.62%)はかなり小さく、検証-Publicギャップ(0.0299)がこれまでの全提出中で最大だった。
TF-IDF由来の45列（3テキスト×15次元のSVD）という高次元・疎な特徴量が、学習2,208件・検証553件という
単一ホールドアウトに過学習気味な検証スコアを出しやすいという仮説を立てた。

## 本ノートブックで行う2つのこと

### 1. TF-IDFの次元調整・頑健性確認
- SVD次元数・`min_df`/`max_features`を変えた3バリエーション（元の15次元、8次元、5次元）を用意する
- **2つの異なる時系列ホールドアウト境界**（従来の80/20と、新たに75/25）の両方で各バリエーションを評価し、
  特定の分割にたまたま適合しているだけでないかを確認する
- 分割によらず安定して良い結果を出すバリエーションを選ぶ

### 2. 有望ブロックの組み合わせ
- 1で選んだ最良のTF-IDFバリエーションに、16_で無害と判明した C（ランク特徴量）・D（四半期/加速度）を
  それぞれ組み合わせて追加改善がないか確認する（80/20の標準splitで、これまでの結果と比較可能な形で実施）

## モデリング方針
14_/16_と同じ、自前CatBoost + Optuna（単一時系列ホールドアウト）パイプラインを使う。

## 実行環境
Google Colab（GPU: T4）を想定。

In [1]:
!pip install -q catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 24.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 26.1 MB/s eta 0:00:00


In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sun Aug  9 08:47:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "17_tfidf_tuning_and_combos"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Output Directory: {OUTPUT_DIR}")

[2026-08-09 08:48:34] [INFO] === [17_tfidf_tuning_and_combos] 実験開始 ===


INFO:17_tfidf_tuning_and_combos:=== [17_tfidf_tuning_and_combos] 実験開始 ===


[2026-08-09 08:48:34] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260809


INFO:17_tfidf_tuning_and_combos:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260809


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-09 08:48:38] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:17_tfidf_tuning_and_combos:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-09 08:48:38] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:17_tfidf_tuning_and_combos:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-09 08:48:38] [INFO] 定着率: 0.5647


INFO:17_tfidf_tuning_and_combos:定着率: 0.5647


[2026-08-09 08:48:38] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:17_tfidf_tuning_and_combos:Train IDs: 2761, Test IDs: 2502


## 1. 特徴量関数の定義（split非依存）

月次集約・カテゴリ変化・欠損値・ドメイン知識・高度統計・クラスター・EDA駆動・上司チーム規模・
テキストTF-IDF（3バリエーション）・ランク・四半期粒度の各特徴量は、目的変数を使わないため
（部署Target Encodingを除く）、時系列split（学習期間/検証期間の境界）に関係なく1回だけ計算すればよい。
部署Target Encodingと職種/等級別グループ平均のみ、15_/16_で判明したリーク（学習期間・検証期間をまたいで
fitすると小規模グループ経由でラベルが漏れる）を避けるため、split後に学習期間のIDだけでfitする。

In [7]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜16_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜16_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜16_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [8]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-09 08:48:39] [INFO] ------------------------------------------------------------


INFO:17_tfidf_tuning_and_combos:------------------------------------------------------------


[2026-08-09 08:48:39] [INFO] split非依存の基本特徴量を生成中...


INFO:17_tfidf_tuning_and_combos:split非依存の基本特徴量を生成中...


[2026-08-09 08:48:39] [INFO] ------------------------------------------------------------


INFO:17_tfidf_tuning_and_combos:------------------------------------------------------------


[2026-08-09 08:55:39] [INFO] split非依存の基本特徴量生成完了


INFO:17_tfidf_tuning_and_combos:split非依存の基本特徴量生成完了


## 2. テキストTF-IDFの3バリエーション

過学習リスクを抑える狙いで、次元数・`min_df`/`max_features`を変えた3バリエーションを用意する。

| バリエーション | max_features | SVD次元数 | min_df |
|---|---|---|---|
| `A_v1`（16_の元設定） | 300 | 15 | 3 |
| `A_v2`（中間的に軽量化） | 200 | 8 | 5 |
| `A_v3`（大幅に軽量化） | 100 | 5 | 8 |

In [9]:
def create_tfidf_svd_features(train_persona, test_persona, col, variant_name, max_features, n_components, min_df, seed=42):
    """文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）"""
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_{variant_name}_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]
TFIDF_VARIANTS = {
    "A_v1": {"max_features": 300, "n_components": 15, "min_df": 3},
    "A_v2": {"max_features": 200, "n_components": 8, "min_df": 5},
    "A_v3": {"max_features": 100, "n_components": 5, "min_df": 8},
}

logger.info("テキストTF-IDF+SVD特徴量（3バリエーション）を生成中...")
tfidf_train_frames = {}
tfidf_test_frames = {}
for variant_name, params in TFIDF_VARIANTS.items():
    train_list, test_list = [], []
    for col in TEXT_COLS:
        tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, variant_name, seed=SEED, **params)
        logger.info(f"{variant_name}/{col}: SVD累積寄与率={explained_var:.3f}")
        train_list.append(tr)
        test_list.append(te)
    tfidf_train_frames[variant_name] = train_list
    tfidf_test_frames[variant_name] = test_list

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-09 08:55:40] [INFO] テキストTF-IDF+SVD特徴量（3バリエーション）を生成中...


INFO:17_tfidf_tuning_and_combos:テキストTF-IDF+SVD特徴量（3バリエーション）を生成中...


[2026-08-09 08:55:41] [INFO] A_v1/入社時メモ: SVD累積寄与率=0.760


INFO:17_tfidf_tuning_and_combos:A_v1/入社時メモ: SVD累積寄与率=0.760


[2026-08-09 08:55:45] [INFO] A_v1/上司からのフィードバック: SVD累積寄与率=0.360


...

INFO:17_tfidf_tuning_and_combos:A_v3/上司からのフィードバック: SVD累積寄与率=0.275


[2026-08-09 08:56:03] [INFO] A_v3/同僚からのフィードバック: SVD累積寄与率=0.327


INFO:17_tfidf_tuning_and_combos:A_v3/同僚からのフィードバック: SVD累積寄与率=0.327


[2026-08-09 08:56:03] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:17_tfidf_tuning_and_combos:テキストTF-IDF+SVD特徴量生成完了


## 3. ランク特徴量（C）・四半期/加速度特徴量（D）

16_のアブレーションで単体ではほぼ無風だった2ブロック。TF-IDFとの組み合わせテスト用に、そのまま流用する。

In [10]:
def create_rank_features(df, group_cols, value_cols):
    out = pd.DataFrame(index=df.index)
    out[ID_COL] = df[ID_COL].values
    for group_col in group_cols:
        for value_col in value_cols:
            if value_col in df.columns and group_col in df.columns:
                out[f"{value_col}_rank_by_{group_col}"] = df.groupby(group_col)[value_col].rank(pct=True)
    return out

def create_quarterly_features(monthly_df, employee_ids, metrics):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

QUARTERLY_METRICS = ["残業時間", "欠勤日数", "月例給与_円", "360度評価_親和度", "有給取得日数", "研修時間"]

logger.info("ランク特徴量・四半期/加速度特徴量を生成中...")

train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_rank_persona = create_rank_features(train_persona, ["初期等級", "入社四半期", "初期職種"], ["初任給_円"])
test_rank_persona = create_rank_features(test_persona, ["初期等級", "入社四半期", "初期職種"], ["初任給_円"])

train_quarterly = create_quarterly_features(train_monthly, train_ids, QUARTERLY_METRICS)
test_quarterly = create_quarterly_features(test_monthly, test_ids, QUARTERLY_METRICS)

logger.info(f"rank: Train {train_rank_persona.shape}, Test {test_rank_persona.shape}")
logger.info(f"quarterly: Train {train_quarterly.shape}, Test {test_quarterly.shape}")

[2026-08-09 08:56:03] [INFO] ランク特徴量・四半期/加速度特徴量を生成中...


INFO:17_tfidf_tuning_and_combos:ランク特徴量・四半期/加速度特徴量を生成中...


[2026-08-09 08:57:21] [INFO] rank: Train (2761, 4), Test (2502, 4)


INFO:17_tfidf_tuning_and_combos:rank: Train (2761, 4), Test (2502, 4)


[2026-08-09 08:57:21] [INFO] quarterly: Train (2761, 31), Test (2502, 31)


INFO:17_tfidf_tuning_and_combos:quarterly: Train (2761, 31), Test (2502, 31)


## 4. Persona単位の基本特徴量（split非依存）

In [11]:
logger.info("Persona単位の基本特徴量を生成中...")
text_cols = TEXT_COLS
train_persona["text_total_chars"] = train_persona[text_cols].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[text_cols].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
for col in text_cols:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-09 08:57:21] [INFO] Persona単位の基本特徴量を生成中...


INFO:17_tfidf_tuning_and_combos:Persona単位の基本特徴量を生成中...


[2026-08-09 08:57:21] [INFO] Persona単位の基本特徴量処理完了


INFO:17_tfidf_tuning_and_combos:Persona単位の基本特徴量処理完了


## 5. split依存の特徴量をまとめる関数

部署Target Encoding（`dept_target_enc`）と職種別乖離・給与相対化特徴量は、目的変数
（または大規模グループとはいえ一貫性のため）学習期間のIDのみでfitする必要があるため、
split（学習期間/検証期間の境界）ごとに再計算する関数にまとめる。

In [12]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    """初期部署IDのKFold + スムージング付きTarget Encoding（15_/16_の修正版と同一ロジック）
    fit_ids: 学習期間のID集合のみでマップを構築する
    """
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, tfidf_variant=None, extra_groups=None):
    """指定した分割比率で、split依存の特徴量を計算し、学習/検証データを組み立てる

    split_ratio: 学習期間の割合（例: 0.8 なら先頭80%が学習期間）
    tfidf_variant: TFIDF_VARIANTSのキー名（Noneならテキスト特徴量を含めない）
    extra_groups: "rank" / "quarterly" を含むリスト（Noneなら含めない）
    """
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")

    # 職種別乖離・給与相対化（学習期間のみでグループ平均を計算）
    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df in [tf, ttf]:
        for m in job_dev_metrics:
            df[f"{m}_job_deviation"] = df[m] - df["初期職種"].map(job_means[m])
        df["研修時間_職種比"] = df["研修時間_mean"] / df["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df["研修時間_区分比"] = df["研修時間_mean"] / df["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df["初任給_等級内偏差"] = df["初任給_円"] - df["初期等級"].map(grade_salary_mean)
        df["初任給_区分内偏差"] = df["初任給_円"] - df["入社区分"].map(category_salary_mean)
        df["月例給与_等級内偏差"] = df["月例給与_円_mean"] - df["初期等級"].map(grade_monthly_salary_mean)

    # オプションのブロックをマージ
    if tfidf_variant is not None:
        for trdf in tfidf_train_frames[tfidf_variant]:
            tf = tf.merge(trdf, on=ID_COL, how="left")
        for tedf in tfidf_test_frames[tfidf_variant]:
            ttf = ttf.merge(tedf, on=ID_COL, how="left")
    if extra_groups and "rank" in extra_groups:
        tf = tf.merge(train_rank_persona, on=ID_COL, how="left")
        ttf = ttf.merge(test_rank_persona, on=ID_COL, how="left")
    if extra_groups and "quarterly" in extra_groups:
        tf = tf.merge(train_quarterly, on=ID_COL, how="left")
        ttf = ttf.merge(test_quarterly, on=ID_COL, how="left")

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    return ag_train, ag_tuning, ttf

print("✅ prepare_split関数定義完了")

✅ prepare_split関数定義完了


## 6. CatBoost + Optuna 実行関数（14_実験Aと同一パイプライン）

In [13]:
def run_catboost_config(ag_train_data, ag_tuning_data, test_features, config_label, n_trials=25):
    feature_cols = [c for c in ag_train_data.columns if c not in ["入社日", TARGET_COL]]
    cat_cols = [c for c in feature_cols if ag_train_data[c].dtype == "object"]

    X_tr = ag_train_data[feature_cols].fillna(-999)
    y_tr = ag_train_data[TARGET_COL]
    X_va = ag_tuning_data[feature_cols].fillna(-999)
    y_va = ag_tuning_data[TARGET_COL]

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000,
            "random_seed": SEED,
            "verbose": False,
            "cat_features": cat_cols,
            "early_stopping_rounds": 50,
            "task_type": "GPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        preds = model.predict_proba(X_va)[:, 1]
        return log_loss(y_va, preds)

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)

    best_params = study.best_params
    final_model = cb.CatBoostClassifier(
        **best_params, iterations=3000, random_seed=SEED, verbose=False,
        cat_features=cat_cols, early_stopping_rounds=100, task_type="GPU",
    )
    final_model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)

    val_preds = final_model.predict_proba(X_va)[:, 1]
    val_score = log_loss(y_va, val_preds)

    X_test = test_features[feature_cols].fillna(-999)
    test_preds = final_model.predict_proba(X_test)[:, 1]
    sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    sub = pd.DataFrame({ID_COL: test_features.index, TARGET_COL: test_preds})
    sub.to_csv(sub_path, index=False, header=False)

    logger.info(f"[{config_label}] n_features={len(feature_cols)}, val_score={val_score:.6f}")
    return {"config": config_label, "n_features": len(feature_cols), "val_score": val_score, "submission_path": str(sub_path)}

print("✅ run_catboost_config関数定義完了")

✅ run_catboost_config関数定義完了


## 7. フェーズ1: TF-IDFバリエーション × 2つのsplitで頑健性確認

`split=0.8`（従来通り、直近20%が検証期間）と`split=0.75`（直近25%が検証期間、より大きい検証集合）の
両方で、`baseline`（TF-IDFなし）・`A_v1`・`A_v2`・`A_v3`を評価する。分割によらず安定して良い
バリエーションを選ぶ。

In [14]:
SPLIT_RATIOS = {"split_80_20": 0.8, "split_75_25": 0.75}
PHASE1_CONFIGS = ["baseline", "A_v1", "A_v2", "A_v3"]

phase1_results = []
for split_name, ratio in SPLIT_RATIOS.items():
    for config in PHASE1_CONFIGS:
        tfidf_variant = None if config == "baseline" else config
        logger.info(f"=== {split_name} / {config} ===")
        ag_train_data, ag_tuning_data, test_features_full = prepare_split(ratio, tfidf_variant=tfidf_variant)
        result = run_catboost_config(ag_train_data, ag_tuning_data, test_features_full, f"{split_name}_{config}", n_trials=25)
        result["split"] = split_name
        result["config"] = config
        phase1_results.append(result)

phase1_df = pd.DataFrame(phase1_results)
phase1_pivot = phase1_df.pivot(index="config", columns="split", values="val_score")
phase1_pivot["mean"] = phase1_pivot.mean(axis=1)
phase1_pivot["std"] = phase1_pivot[["split_80_20", "split_75_25"]].std(axis=1)
phase1_pivot = phase1_pivot.sort_values("mean")

logger.info("=" * 60)
logger.info("フェーズ1結果（検証Log Loss、行=config, 列=split）")
logger.info("=" * 60)
logger.info("\n" + phase1_pivot.to_string())

print("\n■ フェーズ1結果:")
print(phase1_pivot.to_string())

[2026-08-09 08:57:22] [INFO] === split_80_20 / baseline ===


INFO:17_tfidf_tuning_and_combos:=== split_80_20 / baseline ===


[2026-08-09 09:10:53] [INFO] [split_80_20_baseline] n_features=314, val_score=0.550999


INFO:17_tfidf_tuning_and_combos:[split_80_20_baseline] n_features=314, val_score=0.550999


[2026-08-09 09:10:53] [INFO] === split_80_20 / A_v1 ===


...

[2026-08-09 11:01:29] [INFO] ============================================================


INFO:17_tfidf_tuning_and_combos:============================================================


[2026-08-09 11:01:29] [INFO] 
split     split_75_25  split_80_20      mean       std
config                                                
A_v1         0.552253     0.535388  0.543820  0.011925
A_v3         0.556120     0.543482  0.549801  0.008936
A_v2         0.558283     0.543362  0.550822  0.010551
baseline     0.557194     0.550999  0.554097  0.004381


INFO:17_tfidf_tuning_and_combos:
split     split_75_25  split_80_20      mean       std
config                                                
A_v1         0.552253     0.535388  0.543820  0.011925
A_v3         0.556120     0.543482  0.549801  0.008936
A_v2         0.558283     0.543362  0.550822  0.010551
baseline     0.557194     0.550999  0.554097  0.004381



■ フェーズ1結果:
split     split_75_25  split_80_20      mean       std
config                                                
A_v1         0.552253     0.535388  0.543820  0.011925
A_v3         0.556120     0.543482  0.549801  0.008936
A_v2         0.558283     0.543362  0.550822  0.010551
baseline     0.557194     0.550999  0.554097  0.004381


## 8. フェーズ2: 最良TF-IDFバリエーション + ランク特徴量(C) / 四半期特徴量(D)

フェーズ1で「2つのsplitの平均が最良、かつ2つのsplit間の差(std)が小さい」バリエーションを選び、
`split=0.8`（従来と比較可能な標準split）でC・Dをそれぞれ追加した場合の効果を確認する。

In [15]:
best_variant = phase1_pivot.drop("baseline", errors="ignore")["mean"].idxmin()
logger.info(f"フェーズ1で選ばれた最良バリエーション: {best_variant}")
print(f"■ フェーズ1で選ばれた最良バリエーション: {best_variant}")

phase2_configs = [
    (f"{best_variant}_plus_rank", best_variant, ["rank"]),
    (f"{best_variant}_plus_quarterly", best_variant, ["quarterly"]),
]

phase2_results = []
for label, tfidf_variant, extra_groups in phase2_configs:
    logger.info(f"=== split_80_20 / {label} ===")
    ag_train_data, ag_tuning_data, test_features_full = prepare_split(0.8, tfidf_variant=tfidf_variant, extra_groups=extra_groups)
    result = run_catboost_config(ag_train_data, ag_tuning_data, test_features_full, label, n_trials=25)
    phase2_results.append(result)

phase2_df = pd.DataFrame(phase2_results)

logger.info("=" * 60)
logger.info("フェーズ2結果")
logger.info("=" * 60)
logger.info("\n" + phase2_df.to_string())
print("\n■ フェーズ2結果:")
print(phase2_df.to_string())

[2026-08-09 11:01:29] [INFO] フェーズ1で選ばれた最良バリエーション: A_v1


INFO:17_tfidf_tuning_and_combos:フェーズ1で選ばれた最良バリエーション: A_v1


■ フェーズ1で選ばれた最良バリエーション: A_v1
[2026-08-09 11:01:29] [INFO] === split_80_20 / A_v1_plus_rank ===


INFO:17_tfidf_tuning_and_combos:=== split_80_20 / A_v1_plus_rank ===


[2026-08-09 11:20:51] [INFO] [A_v1_plus_rank] n_features=362, val_score=0.541728


...

[2026-08-09 11:48:10] [INFO] ============================================================


INFO:17_tfidf_tuning_and_combos:============================================================


[2026-08-09 11:48:10] [INFO] 
                config  n_features  val_score                                                                                                      submission_path
0       A_v1_plus_rank         362   0.541728       /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_A_v1_plus_rank.csv
1  A_v1_plus_quarterly         389   0.533663  /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_A_v1_plus_quarterly.csv


INFO:17_tfidf_tuning_and_combos:
                config  n_features  val_score                                                                                                      submission_path
0       A_v1_plus_rank         362   0.541728       /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_A_v1_plus_rank.csv
1  A_v1_plus_quarterly         389   0.533663  /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_A_v1_plus_quarterly.csv



■ フェーズ2結果:
                config  n_features  val_score                                                                                                      submission_path
0       A_v1_plus_rank         362   0.541728       /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_A_v1_plus_rank.csv
1  A_v1_plus_quarterly         389   0.533663  /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_A_v1_plus_quarterly.csv


## 9. 総合結果とまとめ

In [16]:
print("■ フェーズ1（split_80_20のみ抜粋、16_の結果と比較可能）:")
print(phase1_df[phase1_df["split"] == "split_80_20"][["config", "val_score", "submission_path"]].to_string(index=False))
print(f"\n(参考) 16_ baseline: 0.547645 / 16_ A_only: 0.529646（Public 0.559570）")

print("\n■ フェーズ2:")
print(phase2_df[["config", "val_score", "submission_path"]].to_string(index=False))

all_results = pd.concat([
    phase1_df[phase1_df["split"] == "split_80_20"][["config", "val_score", "submission_path"]],
    phase2_df[["config", "val_score", "submission_path"]],
]).sort_values("val_score").reset_index(drop=True)

print("\n■ split_80_20における全設定の総合ランキング:")
print(all_results.to_string(index=False))

logger.info("=" * 60)
logger.info("総合結果")
logger.info("=" * 60)
logger.info("\n" + all_results.to_string())

all_results

■ フェーズ1（split_80_20のみ抜粋、16_の結果と比較可能）:
  config  val_score                                                                                                      submission_path
baseline   0.550999 /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_split_80_20_baseline.csv
    A_v1   0.535388     /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_split_80_20_A_v1.csv
    A_v2   0.543362     /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_split_80_20_A_v2.csv
    A_v3   0.543482     /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_split_80_20_A_v3.csv

(参考) 16_ baseline: 0.547645 / 16_ A_only: 0.529646（Public 0.559570）

■ フェーズ2:
             config  val_score                                                                                                     submission_path
     A_v1_plus_rank   0.541728      /content/drive/

INFO:17_tfidf_tuning_and_combos:============================================================


[2026-08-09 11:48:10] [INFO] 総合結果


INFO:17_tfidf_tuning_and_combos:総合結果


[2026-08-09 11:48:10] [INFO] ============================================================


INFO:17_tfidf_tuning_and_combos:============================================================


[2026-08-09 11:48:10] [INFO] 
                config  val_score                                                                                                       submission_path
0  A_v1_plus_quarterly   0.533663   /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_A_v1_plus_quarterly.csv
1                 A_v1   0.535388      /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_split_80_20_A_v1.csv
2       A_v1_plus_rank   0.541728        /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_A_v1_plus_rank.csv
3                 A_v2   0.543362      /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_split_80_20_A_v2.csv
4                 A_v3   0.543482      /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_split_80_20_A_v3.csv
5             baseline   0.550999  /content/drive/MyDrive/

INFO:17_tfidf_tuning_and_combos:
                config  val_score                                                                                                       submission_path
0  A_v1_plus_quarterly   0.533663   /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_A_v1_plus_quarterly.csv
1                 A_v1   0.535388      /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_split_80_20_A_v1.csv
2       A_v1_plus_rank   0.541728        /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_A_v1_plus_rank.csv
3                 A_v2   0.543362      /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_split_80_20_A_v2.csv
4                 A_v3   0.543482      /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_17_tfidf_tuning_and_combos_split_80_20_A_v3.csv
5             baseline   0.550999  /content/drive/MyDri

,config,val_score,submission_path
0,A_v1_plus_quarterly,0.533663,/content/drive/MyDrive/jaggle_2026/data/output...
1,A_v1,0.535388,/content/drive/MyDrive/jaggle_2026/data/output...
2,A_v1_plus_rank,0.541728,/content/drive/MyDrive/jaggle_2026/data/output...
3,A_v2,0.543362,/content/drive/MyDrive/jaggle_2026/data/output...
4,A_v3,0.543482,/content/drive/MyDrive/jaggle_2026/data/output...
5,baseline,0.550999,/content/drive/MyDrive/jaggle_2026/data/output...


## 10. 解釈のポイント

1. **フェーズ1のstd列**: 2つのsplit間で検証スコアの差が大きいバリエーションは、特定の分割に
   過学習気味である可能性が高い。meanが良くてもstdが大きい場合は注意。
2. **16_のA_only（0.529646, Public 0.559570）と本ノートブックのA_v1（split_80_20）を比較**し、
   同じ設定で再現するか確認する（Optuna試行数・seedは同一のはずなので、ほぼ一致するはず）。
3. **フェーズ2でC・Dを追加して改善するか**を確認する。改善しなければ、TF-IDFのみを採用する。
4. 最終的に選んだ設定の提出ファイルをKaggleに提出し、Publicスコアで確認する。
   結果が出たら`submit_result_report.md`に追記する。